# HCNNG

## Clerc-large

In [ ]:
# first prepare data (1000 query)

cd svr-baselines

mkdir -p /data/ali/baseline-data/clerc-large-single/inputs
mkdir -p /data1/chenyifeng/MultiVector-Backup/svr-baselines/runs/clerc-large-single/hcnng
mkdir -p /data1/chenyifeng/MultiVector-Backup/svr-baselines/runs/clerc-large-single/hnswlib
mkdir -p /data1/chenyifeng/MultiVector-Backup/svr-baselines/runs/clerc-large-single/logs

python3 /data1/chenyifeng/MultiVector-Backup/svr-baselines/runs/prepare_clerc_large_single.py \
  --dataset-dir </data/ali/clerc-large-single> \
  --output-dir </data/ali/baseline-data/clerc-large-single/inputs> \
  --num-queries 1000 \
  --dataset-prefix clerc-large-single

In [ ]:
# One-time Build HCNNG

cd hcnng
g++ hcnng.cpp -o hcnng -std=c++11 -fopenmp -O3
g++ search.cpp -o search -std=c++11 -fopenmp -O3

In [ ]:
# HCNNG index build

/data1/chenyifeng/MultiVector-Backup/svr-baselines/hcnng/hcnng \
  /data/ali/clerc-large-single/clerc-large-single_base.fvecs \
  1000 \
  20 \
  /data/ali/baseline-data/clerc-large-single/hcnng/index.ivecs

In [ ]:
# HCNNG query run

/data1/chenyifeng/MultiVector-Backup/svr-baselines/hcnng/search \
  /data/ali/clerc-large-single/clerc-large-single_base.fvecs \
  /data/ali/baseline-data/clerc-large-single/inputs/query_1000.fvecs \
  /data/ali/baseline-data/clerc-large-single/inputs/groundtruth_1000.ivecs \
  /data/ali/baseline-data/clerc-large-single/hcnng/index.ivecs \
  100 \
  2000 \
  /data1/chenyifeng/MultiVector-Backup/svr-baselines/runs/clerc-large-single/hcnng/ans_k100_scored.tsv \
  /data1/chenyifeng/MultiVector-Backup/svr-baselines/runs/clerc-large-single/hcnng/qps_k100_scored.tsv \
  > /data1/chenyifeng/MultiVector-Backup/svr-baselines/runs/clerc-large-single/logs/hcnng_k100_scored.log 2>&1


Important parameter:

- `max_calc=2000`
  - this is HCNNG's search budget
  - it limits how many vectors are explored / distance calculations are spent per query
  - larger `max_calc` usually improves recall and lowers QPS

In [ ]:
# HCNNG Evaluate Recall@10 and Recall@100

python3 /home/ali/SVR-baselines/runs/eval_recall.py \
  --groundtruth /data/ali/baseline-data/clerc-large-single/inputs/groundtruth_1000.ivecs \
  --results /home/ali/SVR-baselines/runs/clerc-large-single/hcnng/ans_k100_scored.tsv \
  --k 10

python3 /home/ali/SVR-baselines/runs/eval_recall.py \
  --groundtruth /data/ali/baseline-data/clerc-large-single/inputs/groundtruth_1000.ivecs \
  --results /home/ali/SVR-baselines/runs/clerc-large-single/hcnng/ans_k100_scored.tsv \
  --k 100

In [ ]:
# BEIR-style evaluation

python3 /home/ali/SVR-baselines/runs/eval_beir_metrics.py \
  --groundtruth /data/ali/baseline-data/clerc-large-single/inputs/groundtruth_1000.ivecs \
  --results /home/ali/SVR-baselines/runs/clerc-large-single/hcnng/ans_k100_scored.tsv \
  --k-values 1 3 5 10 100 \
  --output-json /home/ali/SVR-baselines/runs/clerc-large-single/hcnng/beir_metrics_k100_scored.json

python3 /home/ali/SVR-baselines/runs/eval_beir_metrics.py \
  --groundtruth /data/ali/baseline-data/clerc-large-single/inputs/groundtruth_1000.ivecs \
  --results /home/ali/SVR-baselines/runs/clerc-large-single/hnswlib/ans_k100_scored.tsv \
  --k-values 1 3 5 10 100 \
  --output-json /home/ali/SVR-baselines/runs/clerc-large-single/hnswlib/beir_metrics_k100_scored.json

python3 /home/ali/SVR-baselines/runs/eval_beir_metrics.py \
  --groundtruth /data/ali/baseline-data/clerc-large-single/inputs/groundtruth_1000.ivecs \
  --results /home/ali/SVR-baselines/runs/clerc-large-single/elpis/ans_k100_np1000_scored.tsv \
  --k-values 1 3 5 10 100 \
  --output-json /home/ali/SVR-baselines/runs/clerc-large-single/elpis/beir_metrics_k100_np1000_scored.json

In [ ]:
# HCNNG QPS

# already write

/home/ali/SVR-baselines/runs/clerc-large-single/hcnng/qps_k100_scored.tsv